In [1]:
import numpy as np
import pandas as pd

TRAIN_LABEL = 'data/labeledTrainData.tsv'
TRAIN_UNLABEL = 'data/unlabeledTrainData.tsv'
TEST = 'data/testData.tsv'

all_data = pd.read_csv(TRAIN_LABEL, header=0, delimiter="\t", quoting=3)
bonus_data = pd.read_csv(TRAIN_UNLABEL, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST, header=0, delimiter="\t", quoting=3)

print(all_data.shape)
print(bonus_data.shape)
print(test.shape)

print(all_data.head(2))

all_data.drop(columns=['id'], inplace=True)
bonus_data.drop(columns=['id'], inplace=True)

(25000, 3)
(50000, 2)
(25000, 2)
         id  sentiment                                             review
0  "5814_8"          1  "With all this stuff going down at the moment ...
1  "2381_9"          1  "\"The Classic War of the Worlds\" by Timothy ...


In [2]:
import re

def clean_text(text):
    text = text.lower()
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

all_data['review'] = all_data['review'].apply(clean_text)
bonus_data['review'] = bonus_data['review'].apply(clean_text)
test['review'] = test['review'].apply(clean_text)

test['id'] = [x.replace('"', '') for x in test['id']]
print(test.head(2))

         id                                             review
0  12311_10  naturally in a film whos main themes are of mo...
1    8348_2  this movie is a disaster within a disaster fil...


In [3]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(all_data, test_size=0.2, random_state=42, shuffle=True)

In [4]:
sentences = []
for review in pd.concat([train['review'], bonus_data['review']]):
    sentences.append(review.split())

print(f'Number of sentences: {len(sentences)}')
print(f'First sentence: {sentences[0]}')

Number of sentences: 70000
First sentence: ['this', 'movie', 'is', 'just', 'plain', 'dumbbr', 'br', 'from', 'the', 'casting', 'of', 'ralph', 'meeker', 'as', 'mike', 'hammer', 'to', 'the', 'fatuous', 'climax', 'the', 'film', 'is', 'an', 'exercise', 'in', 'wooden', 'predictabilitybr', 'br', 'mike', 'hammer', 'is', 'one', 'of', 'detective', 'fictions', 'true', 'sociopaths', 'unlike', 'marlow', 'and', 'spade', 'who', 'put', 'pieces', 'together', 'to', 'solve', 'the', 'mystery', 'hammer', 'breaks', 'things', 'apart', 'to', 'get', 'to', 'the', 'truth', 'this', 'film', 'turns', 'hammer', 'into', 'a', 'boob', 'by', 'surrounding', 'him', 'with', 'bad', 'guys', 'who', 'are', 'well', 'too', 'dumb', 'to', 'get', 'away', 'with', 'anything', 'one', 'is', 'so', 'poorly', 'drawn', 'that', 'he', 'succumbs', 'to', 'a', 'popcorn', 'attackbr', 'br', 'other', 'parts', 'of', 'the', 'movie', 'are', 'right', 'out', 'of', 'the', 'three', 'stooges', 'play', 'book', 'veldas', 'dance', 'at', 'the', 'barre', 'for'

In [5]:
from gensim.models import Word2Vec
from tqdm import tqdm

NUM_FEATURES = 300
EPOCHS = 10
SAMPLE = 1e-3
MIN_COUNT = 10
WINDOW = 10
WORKERS = 40

w2v_model = Word2Vec(
    sentences=sentences,
    workers=WORKERS,
    vector_size=NUM_FEATURES,# Dimensionality of the word vectors
    window=WINDOW,      # Maximum distance between the current and predicted word within a sentence
    min_count=MIN_COUNT,   # Ignores all words with total frequency lower than this
    sample=SAMPLE,    # Threshold for downsampling high-frequency words
    epochs=EPOCHS
)

w2v_model.init_sims(replace=True)  # Lock the model to prevent further training

print(w2v_model.wv.doesnt_match("man woman child kitchen".split()))

kitchen


C:\Users\ngbac\AppData\Local\Temp\ipykernel_34172\1870306070.py:21: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.init_sims(replace=True)  # Lock the model to prevent further training


In [6]:
def makeFeatureVec(words, model, num_features):
    vectors = [model.wv[word] for word in words if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(num_features, dtype="float32")

def getAvgFeatureVecs(reviews, model, num_features):
    review_vecs = np.zeros((len(reviews), num_features), dtype="float32")
    for i, review in enumerate(tqdm(reviews, desc="Processing reviews")):
        review_vecs[i] = makeFeatureVec(review.split(), model, num_features)
    return review_vecs

In [7]:
X_train = getAvgFeatureVecs(train['review'], w2v_model, NUM_FEATURES)
y_train = train['sentiment'].values

X_val = getAvgFeatureVecs(val['review'], w2v_model, NUM_FEATURES)
y_val = val['sentiment'].values

X_test = getAvgFeatureVecs(test['review'], w2v_model, NUM_FEATURES)

Processing reviews: 100%|██████████| 25000/25000 [00:05<00:00, 4257.93it/s]


In [8]:
print(X_train.shape)

(20000, 300)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model = RandomForestClassifier(random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)

auc = roc_auc_score(y_val, val_preds)
print(f'Validation AUC: {auc}')

Validation AUC: 0.8360339693220681


In [10]:
sentences = []
for review in pd.concat([all_data['review'], bonus_data['review']]):
    sentences.append(review.split())

w2v_model = Word2Vec(
    sentences=sentences,
    workers=WORKERS,
    vector_size=NUM_FEATURES,# Dimensionality of the word vectors
    window=WINDOW,      # Maximum distance between the current and predicted word within a sentence
    min_count=MIN_COUNT,   # Ignores all words with total frequency lower than this
    sample=SAMPLE,    # Threshold for downsampling high-frequency words
    epochs=EPOCHS
)

w2v_model.init_sims(replace=True)  # Lock the model to prevent further training

X_train = getAvgFeatureVecs(all_data['review'], w2v_model, NUM_FEATURES)
y_train = all_data['sentiment'].values
X_test = getAvgFeatureVecs(test['review'], w2v_model, NUM_FEATURES)

model = RandomForestClassifier(random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

test_preds = model.predict(X_test)

C:\Users\ngbac\AppData\Local\Temp\ipykernel_34172\1948118994.py:15: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.init_sims(replace=True)  # Lock the model to prevent further training
Processing reviews: 100%|██████████| 25000/25000 [00:06<00:00, 3577.18it/s]


In [11]:
submission = pd.DataFrame({
    'id': test['id'],
    'sentiment': test_preds
})

submission.to_csv('submissions/w2v_rf.csv', index=False)